# Taller: Introducción a Machine Learning

## 1. Introducción
El aprendizaje automático (**Machine Learning, ML**) es una rama de la inteligencia artificial que permite a las computadoras **aprender patrones a partir de datos** para realizar predicciones o clasificaciones sin ser programadas explícitamente.  
En este taller trabajaremos con un flujo completo de entrenamiento de un modelo clásico de ML: las **Máquinas de Vectores de Soporte (SVM, Support Vector Machines)**, ampliamente utilizadas en aplicaciones biomédicas por su capacidad de separar clases de manera robusta, incluso en espacios de alta dimensión.

El objetivo es que comprendas, paso a paso, cómo preparar los datos, entrenar un modelo, evaluar su rendimiento y visualizar los resultados. No solo aprenderás la teoría, sino que también **ejecutarás código práctico**, explorando cómo cambian los resultados al variar hiperparámetros o aumentar las iteraciones de entrenamiento.

## 2. Objetivos
1. **Carga y exploración de datos**: comprender la estructura y características principales del dataset.  
2. **División de datos (train/test)**: entender su importancia para evaluar la generalización del modelo.  
3. **Curvas de aprendizaje (SVM lineal)**: identificar problemas de sesgo y varianza.  
4. **Entrenamiento con SVM RBF**: aplicar un kernel no lineal y ajustar hiperparámetros.  
5. **Entrenamiento iterativo**: observar cómo mejora el desempeño con más épocas.  
6. **Evaluación del modelo**: interpretar métricas y una matriz de confusión.  
7. **Visualización en 2D con PCA**: proyectar los datos y observar las fronteras de decisión.  

## 3. Dataset


En este taller utilizaremos **MedMNIST**, una colección estandarizada de imágenes biomédicas en formato *MNIST-like*.  
Fue desarrollada para facilitar la investigación y la enseñanza en **análisis de imágenes médicas** y **machine learning**, ofreciendo un punto de partida ligero y accesible.

![](/Users/carlos/Documents/github/Talleres_Diplomado_iHealth/img/medmnist.png)


## 3.1 Cargar datos


En **MedMNIST**, los datasets se cargan fácilmente con su API oficial.  
Por ejemplo, para el caso de **PneumoniaMNIST**:

```python
from medmnist import PneumoniaMNIST
from matplotlib import pyplot as plt

# Cargamos el set de entrenamiento
train_dataset = PneumoniaMNIST(split="train", download=True)

# Exploramos algunos elementos
for image, label in train_dataset:
    print(image.shape, label)
    break
```

### División de los datos

Al igual que en otros datasets de aprendizaje automático, los datos no se entregan como un único conjunto, sino divididos en particiones:
* Train (entrenamiento): datos que el modelo ve y utiliza para aprender los parámetros.
 * Test (prueba): datos reservados, que nunca se usan en el entrenamiento, y que sirven para medir la capacidad del modelo de generalizar a información nueva.
 * Validation (validación): en algunos casos (como PneumoniaMNIST), existe un tercer conjunto que se utiliza durante el desarrollo para ajustar hiperparámetros o comparar variantes del modelo sin tocar aún el test final.

¿Por qué es importante esta separación?
 * Evita sobreajuste (overfitting): si el modelo se evaluara únicamente en los datos de entrenamiento, obtendría métricas irreales, ya que estaría “memorizando” los ejemplos.
 * Permite elegir modelos: el conjunto de validación ayuda a decidir qué parámetros o configuraciones funcionan mejor antes de la evaluación final.
 * Asegura una medición justa: el conjunto de test funciona como un examen final, garantizando que el rendimiento reportado refleja la capacidad de generalización a nuevos datos.



## 3.2 Visualización

#### **Ejercicio 1** 


In [ ]:
import sys, importlib, types, numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, learning_curve, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay
from sklearn.svm import SVC
from sklearn.linear_model import SGDClassifier
import warnings; warnings.filterwarnings('ignore')
%matplotlib inline

sys.path.append('/mnt/data')

ae = importlib.import_module('analisis_exploratorio') if 'analisis_exploratorio' not in sys.modules else sys.modules['analisis_exploratorio']
pca_mod = importlib.import_module('pca') if 'pca' not in sys.modules else sys.modules['pca']
th = importlib.import_module('test_hipotesis') if 'test_hipotesis' not in sys.modules else sys.modules['test_hipotesis']
reg = importlib.import_module('regresion') if 'regresion' not in sys.modules else sys.modules['regresion']

def get_func(module, name, default):
    return getattr(module, name) if hasattr(module, name) else default


### Funciones por defecto (fallback)
Si tus módulos ya tienen funciones equivalentes, se usarán automáticamente. Si no, activamos estos **fallbacks**.

In [ ]:
# ---- Fallbacks (se usan solo si no existen en los módulos) ----
def _eda_resumen(df):
    display(df.head())
    display(df.describe().T)

def _split_scale(X, y, test_size=0.2, random_state=42):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state, stratify=y
    )
    sc = StandardScaler()
    X_train_sc = sc.fit_transform(X_train)
    X_test_sc  = sc.transform(X_test)
    return X_train_sc, X_test_sc, y_train, y_test, sc

def _plot_learning_curve(est, X, y, title='Curva de aprendizaje'):
    train_sizes, train_scores, test_scores = learning_curve(
        est, X, y, cv=5, n_jobs=-1, train_sizes=np.linspace(0.1, 1.0, 8), scoring='accuracy'
    )
    plt.figure(figsize=(6,4))
    plt.plot(train_sizes, train_scores.mean(axis=1), marker='o', label='Train')
    plt.plot(train_sizes, test_scores.mean(axis=1), marker='s', label='CV')
    plt.xlabel('Tamaño de entrenamiento'); plt.ylabel('Accuracy'); plt.title(title)
    plt.grid(True); plt.legend(); plt.show()

def _gridsearch_svc_rbf(X, y, param_grid=None):
    if param_grid is None:
        param_grid = {'C':[0.1,1,10,100], 'gamma':['scale',0.1,0.01,0.001]}
    gs = GridSearchCV(SVC(kernel='rbf'), param_grid, cv=5, n_jobs=-1, scoring='accuracy')
    gs.fit(X, y)
    return gs

def _train_iterative_svm(Xtr, ytr, Xte, yte, n_epochs=10, alpha=1e-4):
    clf = SGDClassifier(loss='hinge', alpha=alpha, random_state=42)
    acc_tr, acc_te = [], []
    classes = np.unique(ytr)
    for _ in range(n_epochs):
        clf.partial_fit(Xtr, ytr, classes=classes)
        acc_tr.append(accuracy_score(ytr, clf.predict(Xtr)))
        acc_te.append(accuracy_score(yte, clf.predict(Xte)))
    return clf, np.array(acc_tr), np.array(acc_te)

def _plot_confusion(y_true, y_pred, title='Matriz de confusión'):
    ConfusionMatrixDisplay.from_predictions(y_true, y_pred)
    plt.title(title); plt.show()

def _plot_decision_boundary_2d(model, Xtr, ytr, title='Frontera de decisión (PCA 2D)'):
    pca = PCA(n_components=2)
    X2d = pca.fit_transform(Xtr)
    from sklearn.base import clone
    m2d = clone(model)
    m2d.fit(X2d, ytr)
    x_min, x_max = X2d[:,0].min()-1, X2d[:,0].max()+1
    y_min, y_max = X2d[:,1].min()-1, X2d[:,1].max()+1
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200), np.linspace(y_min, y_max, 200))
    Z = m2d.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    plt.figure(figsize=(6,5))
    plt.contourf(xx, yy, Z, alpha=0.2)
    plt.scatter(X2d[:,0], X2d[:,1], c=ytr, edgecolor='k', s=30)
    plt.xlabel('PC1'); plt.ylabel('PC2'); plt.title(title); plt.grid(True); plt.show()

# Mapear funciones: primero buscar en módulos, si no existen usar fallback
eda_resumen = get_func(ae, 'eda_resumen', _eda_resumen)
split_scale = get_func(ae, 'split_scale', _split_scale)
plot_learning_curve_mod = get_func(reg, 'plot_learning_curve', _plot_learning_curve)
gridsearch_svc_rbf = get_func(reg, 'gridsearch_svc_rbf', _gridsearch_svc_rbf)
train_iterative_svm = get_func(reg, 'train_iterative_svm', _train_iterative_svm)
plot_confusion_mod = get_func(reg, 'plot_confusion', _plot_confusion)
plot_decision_boundary_2d = get_func(pca_mod, 'plot_decision_boundary_2d', _plot_decision_boundary_2d)


## 1) Carga y EDA
**Ejercicio 1:** ejecuta la EDA y comenta las variables más relevantes.

In [ ]:
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name='target')
print(X.shape, y.shape)
eda_resumen(X)
print('Clases:', dict(zip(data.target_names, np.bincount(y))))


## 2) Split + Escalado
**Ejercicio 2:** ajusta `test_size` si lo consideras necesario.

In [ ]:
X_train_sc, X_test_sc, y_train, y_test, scaler = split_scale(X, y, test_size=0.2, random_state=42)
X_train_sc.shape, X_test_sc.shape


## 3) Curvas de aprendizaje (SVM lineal)
**Ejercicio 3:** genera la curva y discute si hay under/overfitting.

In [ ]:
plot_learning_curve_mod(SVC(kernel='linear', C=1.0), X_train_sc, y_train, title='Curva de aprendizaje (SVM lineal)')


## 4) SVM RBF + GridSearch
**Ejercicio 4:** modifica la rejilla y compara resultados, tiempo y accuracy.

In [ ]:
param_grid = {'C':[0.1,1,10,100], 'gamma':['scale', 0.1, 0.01, 0.001]}
gs = gridsearch_svc_rbf(X_train_sc, y_train, param_grid)
best_svc = gs.best_estimator_ if hasattr(gs, 'best_estimator_') else gs
print('Mejores params:', getattr(gs, 'best_params_', 'N/A'))
print('Mejor CV acc:', getattr(gs, 'best_score_', 'N/A'))
from sklearn.metrics import accuracy_score
print('Test acc:', accuracy_score(y_test, best_svc.predict(X_test_sc)))


## 5) Entrenamiento iterativo (control de épocas)
Usamos `SGDClassifier(loss='hinge')` para simular SVM lineal con entrenamiento por épocas visibles.

**Ejercicio 5:** cambia `n_epochs` y observa la evolución.

In [ ]:
n_epochs = 25  # <-- CAMBIA ESTE VALOR
clf_sgd, acc_tr_hist, acc_te_hist = train_iterative_svm(X_train_sc, y_train, X_test_sc, y_test, n_epochs=n_epochs)

plt.figure(figsize=(6,4))
plt.plot(range(1, n_epochs+1), acc_tr_hist, marker='o', label='Train')
plt.plot(range(1, n_epochs+1), acc_te_hist, marker='s', label='Test')
plt.xlabel('Épocas'); plt.ylabel('Accuracy'); plt.title('Evolución por épocas (SGDClassifier)')
plt.grid(True); plt.legend(); plt.show()
print('Accuracy final (test):', acc_te_hist[-1])


## 6) Matriz de confusión y reporte
**Ejercicio 6:** elige el modelo (`best_svc` o `clf_sgd`).

In [ ]:
modelo = best_svc  # o: modelo = clf_sgd
y_pred = modelo.predict(X_test_sc)
plot_confusion_mod(y_test, y_pred)
print(classification_report(y_test, y_pred, target_names=load_breast_cancer().target_names))


## 7) Fronteras de decisión (PCA 2D)
**Ejercicio 7:** compara `best_svc` (RBF) vs `clf_sgd` (lineal).

In [ ]:
plot_decision_boundary_2d(best_svc, X_train_sc, y_train, title='Frontera RBF (PCA 2D)')
plot_decision_boundary_2d(clf_sgd, X_train_sc, y_train, title='Frontera lineal (PCA 2D)')


## 8) Opcional
- Añade `class_weight='balanced'` si notas desbalance.
- Reporta **ROC-AUC** y dibuja curva ROC.
- Usa `StratifiedKFold` para CV más controlada.
- Guarda el mejor modelo con `joblib`.
